# EDA 

In [2]:
import pandas as pd 
from pathlib import Path
from sklearn.model_selection import train_test_split
import sweetviz as sv
from pandas import DataFrame

# CREDICT_CARD_PATH  = Path.cwd() / "data" / "creditcardfraud" / "creditcard.csv"
NETWORK_TRAFFIC_PATH = Path.cwd() / "data" / "network_traffic" 
NETWORK_TRAFFIC_DATA = NETWORK_TRAFFIC_PATH / "embedded_system_network_security_dataset.csv"

NAB_PATH = Path.cwd() / "data" / "nab" / "realAWSCloudwatch"
NAB_DATA = NAB_PATH / "realAWSCloudwatch/ec2_cpu_utilization_24ae8d.csv"
print(NAB_DATA)


/Users/mayazwang/Files/00_Learning/_AI_security/Anomaly_detection/data/nab/realAWSCloudwatch/realAWSCloudwatch/ec2_cpu_utilization_24ae8d.csv


### Network Traffic Anomaly Detection Dataset

In [3]:
df_ntad = pd.read_csv(NETWORK_TRAFFIC_DATA)
df_ntad.head(10)

,packet_size,inter_arrival_time,src_port,dst_port,packet_count_5s,mean_packet_size,spectral_entropy,frequency_band_energy,label,protocol_type_TCP,protocol_type_UDP,src_ip_192.168.1.2,src_ip_192.168.1.3,dst_ip_192.168.1.5,dst_ip_192.168.1.6,tcp_flags_FIN,tcp_flags_SYN,tcp_flags_SYN-ACK
0,0.405154,0.620362,62569,443,0.857143,0.0,0.834066,0.534891,0.0,False,True,True,False,False,False,False,False,False
1,0.527559,0.741288,59382,443,0.785714,0.0,0.147196,0.990757,0.0,False,True,False,False,False,True,False,True,False
2,0.226199,0.485116,65484,80,0.285714,0.0,0.855192,0.031781,0.0,False,True,False,False,True,False,False,False,False
3,0.573372,0.450965,51707,53,0.142857,0.0,0.153220,0.169958,0.0,False,False,False,True,False,False,False,False,False
4,0.651396,0.888740,26915,53,0.714286,0.0,0.923916,0.552053,0.0,True,False,False,True,False,False,False,True,False
5,0.568361,0.207250,2619,53,0.928571,0.0,0.113375,0.582346,0.0,True,False,True,False,False,False,False,False,False
6,0.571940,0.363908,14582,53,0.714286,0.0,0.874457,0.760548,0.0,False,False,True,False,False,False,False,True,False
7,0.173228,0.075335,5724,443,0.428571,0.0,0.541747,0.175055,0.0,False,False,True,False,True,False,False,False,True
8,0.967788,0.493461,28474,443,0.071429,0.0,0.330655,0.023906,0.0,False,False,False,False,False,False,True,False,False
9,0.364352,0.434362,22620,443,0.571429,0.0,0.056789,0.172246,0.0,False,True,False,True,False,True,False,False,False


### NAB Dataset

In [4]:
df_nab = pd.read_csv(NAB_DATA)
df_nab.head(10)

,timestamp,value
0,2014-02-14 14:30:00,0.132
1,2014-02-14 14:35:00,0.134
2,2014-02-14 14:40:00,0.134
3,2014-02-14 14:45:00,0.134
4,2014-02-14 14:50:00,0.134
5,2014-02-14 14:55:00,0.134
6,2014-02-14 15:00:00,0.134
7,2014-02-14 15:05:00,0.134
8,2014-02-14 15:10:00,0.066
9,2014-02-14 15:15:00,0.132


## T, V, H split 


In [24]:
# Split data into 70% train, 15% validation, 15% holdout, maintaining class ratios of 'y'

def t_v_h_split(X, y, 
                path: str,
                p_valid: float = 0.15, 
                p_hold: float = 0.15):

    p_test = (p_valid + p_hold)
    p_create_hold = p_hold/p_test

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=p_test, 
        random_state=42, 
        stratify=y 
    )

    X_valid, X_hold, y_valid, y_hold = train_test_split(
            X_test, y_test, 
            test_size=p_create_hold, 
            random_state=42, 
            stratify=y_test
        )

    df_train = pd.concat([X_train, y_train], axis=1)
    df_valid = pd.concat([X_valid, y_valid], axis=1)
    df_hold = pd.concat([X_hold, y_hold], axis=1)

    df_train.to_csv(f"{path}/df_train.csv", index=False)
    df_valid.to_csv(f"{path}/df_valid.csv", index=False)
    df_hold.to_csv(f"{path}/df_hold.csv", index=False)

    return df_train, df_valid, df_hold

In [25]:
df_train, df_valid, df_hold = t_v_h_split(X=df_ntad.drop("label", axis=1), y=df_ntad["label"], path=str(NETWORK_TRAFFIC_PATH))

In [21]:
def check_df(df):
    print(f"Shape: {df.shape}")
    print(df.head())

In [22]:
check_df(df_train)

Shape: (700, 18)
     packet_size  inter_arrival_time  src_port  dst_port  packet_count_5s  \
979     0.130995            0.325352     64796        80         0.714286   
474     0.983536            0.341513     25476       443         0.357143   
379     0.040086            0.812864      7022        80         0.071429   
513     0.928418            0.725507     60573        80         0.000000   
125     0.545455            0.323503     30487        53         0.571429   

     mean_packet_size  spectral_entropy  frequency_band_energy  \
979               0.0          0.836820               0.621469   
474               0.0          0.109997               0.462735   
379               0.0          0.506324               0.734016   
513               0.0          0.914045               0.923464   
125               0.0          0.992607               0.590219   

     protocol_type_TCP  protocol_type_UDP  src_ip_192.168.1.2  \
979              False               True               Fa

In [26]:
check_df(df_valid)

Shape: (150, 18)
     packet_size  inter_arrival_time  src_port  dst_port  packet_count_5s  \
524     0.979241            0.983520     19842       443         0.857143   
768     0.156049            0.350412     54205       443         0.428571   
169     0.231926            0.734841      1785        53         0.500000   
130     0.488189            0.551635      3609        53         0.000000   
435     0.879026            0.147497     21152        80         0.928571   

     mean_packet_size  spectral_entropy  frequency_band_energy  \
524               0.0          0.745524               0.286342   
768               0.0          0.846390               0.250348   
169               0.0          0.775188               0.435533   
130               0.0          0.847939               0.332863   
435               0.0          0.474528               0.997911   

     protocol_type_TCP  protocol_type_UDP  src_ip_192.168.1.2  \
524               True              False                T

In [27]:
check_df(df_hold)

Shape: (150, 18)
     packet_size  inter_arrival_time  src_port  dst_port  packet_count_5s  \
662     0.467430            0.055018     16225        53         0.285714   
91      0.600573            0.866049     47588       443         0.785714   
973     0.619184            0.171404     37053        80         0.428571   
511     0.937008            0.384348     29407        80         0.071429   
93      0.725841            0.582971     18337       443         0.714286   

     mean_packet_size  spectral_entropy  frequency_band_energy  \
662               0.0          0.419732               0.535526   
91                0.0          0.701968               0.078630   
973               0.0          0.426179               0.066045   
511               0.0          0.488388               0.002319   
93                0.0          0.282401               0.765451   

     protocol_type_TCP  protocol_type_UDP  src_ip_192.168.1.2  \
662               True              False               Fa

In [28]:
# compare the dataset
compare = sv.compare(source=df_valid, compare=df_hold, target_feat="label")

# Show the result
compare.show_html(f"{str(NETWORK_TRAFFIC_PATH)}/Compare_valid_v_hold.html")

Feature: label (TARGET)                      |▌         | [  5%]   00:00 -> (00:00 left)findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
Feature: packet_size                         |█         | [ 11%]   00:00 -> (00:02 left)findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal for Roboto, now using 500.
findfont: Failed to find font weight normal

Report /Users/mayazwang/Files/00_Learning/_AI_security/Anomaly_detection/data/network_traffic/Compare_valid_v_hold.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.
